# FAKE NEWS GROUP PROJECT: Group 54
## Preparing for the code

In [ ]:
# Importing the needed modules
import re
import math
import nltk
import time
import numpy as np
import pandas as pd
from scipy.sparse import hstack
import matplotlib.pyplot as plt
from collections import Counter
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import confusion_matrix,  ConfusionMatrixDisplay

In [2]:
# Only needed once
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\bruge\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bruge\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\bruge\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bruge\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## Part 1 - Task 1

In [3]:
# Loading the dataset
data = pd.read_csv('fakenews_sample.csv')
data.head()

,Unnamed: 0,id,domain,type,url,content,scraped_at,inserted_at,updated_at,title,authors,keywords,meta_keywords,meta_description,tags,summary
0,0,141,awm.com,unreliable,http://awm.com/church-congregation-brings-gift...,Sometimes the power of Christmas will make you...,2018-01-25 16:17:44.789555,2018-02-02 01:19:41.756632,2018-02-02 01:19:41.756664,Church Congregation Brings Gift to Waitresses ...,Ruth Harris,NaN,[''],NaN,NaN,NaN
1,1,256,beforeitsnews.com,fake,http://beforeitsnews.com/awakening-start-here/...,AWAKENING OF 12 STRANDS of DNA – “Reconnecting...,2018-01-25 16:17:44.789555,2018-02-02 01:19:41.756632,2018-02-02 01:19:41.756664,AWAKENING OF 12 STRANDS of DNA – “Reconnecting...,Zurich Times,NaN,[''],NaN,NaN,NaN
2,2,700,cnnnext.com,unreliable,http://www.cnnnext.com/video/18526/never-hike-...,Never Hike Alone: A Friday the 13th Fan Film U...,2018-01-25 16:17:44.789555,2018-02-02 01:19:41.756632,2018-02-02 01:19:41.756664,Never Hike Alone - A Friday the 13th Fan Film ...,NaN,NaN,[''],Never Hike Alone: A Friday the 13th Fan Film ...,NaN,NaN
3,3,768,awm.com,unreliable,http://awm.com/elusive-alien-of-the-sea-caught...,"When a rare shark was caught, scientists were ...",2018-01-25 16:17:44.789555,2018-02-02 01:19:41.756632,2018-02-02 01:19:41.756664,Elusive ‘Alien Of The Sea ‘ Caught By Scientis...,Alexander Smith,NaN,[''],NaN,NaN,NaN
4,4,791,bipartisanreport.com,clickbait,http://bipartisanreport.com/2018/01/21/trumps-...,Donald Trump has the unnerving ability to abil...,2018-01-25 16:17:44.789555,2018-02-02 01:19:41.756632,2018-02-02 01:19:41.756664,Trump’s Genius Poll Is Complete & The Results ...,Gloria Christie,NaN,[''],NaN,NaN,NaN


In [4]:
# Function to clean the dataset
def clean_text(text):
    # Empty values
    if pd.isna(text):
        return ""
    
    # lower the text
    text = text.lower()

    # Replace url with a <URL> tag
    text = re.sub(r'https?://\S+|www\.\S+', '<URL>', text)

    # Replace emails with <EMAIL> tag
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '<EMAIL>', text)

    # Replace dates with <DATE> tag
    text = re.sub(r'[0-9]+[a-zA-Z]+', '<DATE>', text)

    # Replace all other numbers with <NUM> tag
    text = re.sub(r'[0-9]+', '<NUM>', text)
    
    # Remove any special charachters
    text = re.sub(r'[^a-z\s<>]', '', text)

    # Remove spaces, tabs and line shifts
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text # Returning the cleaned text

In [5]:
# Applying the cleaning function to the content column of the dataset
data["clean_content"] = data["content"].apply(clean_text)
data[["content", "clean_content"]].head()

# Tokenize
data['tokens'] = data['clean_content'].apply(nltk.word_tokenize)
data[['clean_content', 'tokens']].head()

# Filter tokens 
english_stopwords = set(stopwords.words('english')) # Define the english stopwords
data['filtered_tokens'] = data['tokens'].apply(lambda tokens: [word for word in tokens if word not in english_stopwords])
data[['tokens', 'filtered_tokens']].head()

# Stemming
ps = PorterStemmer() # Creating a stemmer
data['stemmed'] = data['filtered_tokens'].apply(lambda x: [ps.stem(w) if not w.startswith('<') else w for w in x])

In [6]:
# Extracting all words from the tokens, filtered tokens and stemmed lists
all_words_list = [word for sublist in data['tokens'] for word in sublist]
filtered_words_list = [word for sublist in data['filtered_tokens'] for word in sublist]
all_stemmed_words = [word for sublist in data['stemmed'] for word in sublist]

# Finding unique words to create vocabularies
vocab = set(all_words_list)
vocab_filt = set(filtered_words_list)
vocab_stemmed = set(all_stemmed_words)

# Calculate lenght of the vocabularies
vocab_size = len(vocab)
filt_vocab_size = len(vocab_filt)
vocab_stemmed_size = len(vocab_stemmed)

In [ ]:
# Calculating the reduction in vocabulary size after removing stopwords
reduction_filtering = (1 - (filt_vocab_size / vocab_size)) * 100

# Calculating the reduction in vocabulary size after stemming
reduction_stemming = (1 - (vocab_stemmed_size / filt_vocab_size)) * 100

# Printing the statistics
print(f"Number of stopwords: {len(english_stopwords)}")
print(f"The first 15 english stopwords: {list(english_stopwords)[:15]}")
print("-------------------------------------")
print(f"Original Vocabulary Size: {vocab_size}")
print(f"Vocabulary Size after removing Stopwords: {filt_vocab_size} (Reduction: {reduction_filtering:.2f}%)")
print(f"Vocabulary Size after Stemming: {vocab_stemmed_size} (Reduction: {reduction_stemming:.2f}%)")

Number of stopwords: 198
The first 15 english stopwords: ['again', "didn't", 'his', "doesn't", 'ourselves', 'do', "it'll", 'which', 'own', 'about', 'ma', 'other', "it'd", 'from', 'had']
-------------------------------------
Original Vocabulary Size: 16471
Vocabulary Size after removing Stopwords: 16339 (Reduktion: 0.80%)
Vocabulary Size after Stemming: 10926 (Reduktion: 33.13%)


## Part 1 - Task 2

In [ ]:
# Create variable for input file
input_file = "995K_rows.csv"

# Define chunk size
chunk_size = 15000 

# Create a dataset reader
reader = pd.read_csv(input_file, chunksize=chunk_size)

# Create variable for output file
output_file = "processed_995K.csv"

# Initialize global statistics
global_vocab_initial = set()
global_vocab_filtered = set()
global_vocab_stemmed = set()

In [ ]:
# Print to keep track of the loading of the dataset
print(f"Starter processering af {input_file}...")
print(f"Chunk size: {chunk_size}")
print("------------------------------------------")

# Set the start time
start_time = time.time()

for i, chunk in enumerate(reader):
    chunk_start = time.time()
    
    # Apply the cleaning function
    chunk['content_cleaned'] = chunk['content'].apply(clean_text)

    # Tokenize
    chunk['tokens'] = chunk['content_cleaned'].apply(nltk.word_tokenize)
    
    # Update the original vocabulary
    for t_list in chunk['tokens']:
        global_vocab_initial.update(t_list)
    
    # Filter for stopwords
    chunk['filtered_tokens'] = chunk['tokens'].apply(lambda t: [w for w in t if w not in english_stopwords])

    # Update the filtered vocbulary
    for t_list in chunk['filtered_tokens']:
        global_vocab_filtered.update(t_list)
    
    # Stemming
    chunk['stemmed'] = chunk['filtered_tokens'].apply(lambda x: [ps.stem(w) if not w.startswith('<') else w for w in x])

    # Update the stemmed vocabulary
    for t_list in chunk['stemmed']:
        global_vocab_stemmed.update(t_list)
    
    # Save the data to the output file
    is_first = (i == 0)
    chunk.to_csv(output_file, index=False, mode='w' if is_first else 'a', header=is_first)
    
    # Calculate the status
    chunk_duration = time.time() - chunk_start
    total_elapsed = time.time() - start_time
    rows_so_far = (i + 1) * chunk_size
    
    # Print the status to keep track of process
    print(f"Chunk {i+1:3} | Rækker: {rows_so_far:7,} | Tid: {chunk_duration:5.2f}s | Total: {total_elapsed/60:5.2f} min")

In [ ]:
# Find lenght of the datasets
vocab_og = len(global_vocab_initial)
vocab_filt = len(global_vocab_filtered)
vocab_stem = len(global_vocab_stemmed)

# Calculate reduction in vocabulary size
red_filt = (1 - (vocab_filt/vocab_og)) * 100
red_stem = (1 - (vocab_stem/vocab_og)) * 100

# Printing the statistics
print(f"Original Vocabulary Size: {vocab_og}")
print(f"Vocabulary Size after removing Stopwords: {vocab_filt} (Reduction: {red_filt:.2f}%)")
print(f"Vocabulary Size after Stemming: {vocab_stem} (Reduction: {red_stem:.2f}%)")

## Part 1 - Task 3

In [ ]:
# Read the processed datafile
df = pd.read_csv("processed_995K.csv", usecols=['type', 'domain', 'authors', 'content', 'content_cleaned', 'stemmed'])

# Fill out missing values of the most important columns
df['content'] = df['content'].fillna('')
df['content_cleaned'] = df['content_cleaned'].fillna('')
df['stemmed'] = df['stemmed'].fillna('')

In [ ]:
# Count URLs, dates and numbers in the dataset
url_count = df['content'].str.count(r'https?://\S+|www\.\S+').sum()
date_count = df['content'].str.count(r'[0-9]+[a-zA-Z]+').sum()
num_count = df['content'].str.count(r'[0-9]+').sum()

# Print the results
print(f"Total URLs count: {url_count:,.0f}")
print(f"Total dates count: {date_count:,.0f}")
print(f"Total number count: {num_count:,.0f}")

In [ ]:
# Find unique label types (column: type)
unique_labels = df['type'].unique()
print(f"Unique label types: {unique_labels}")

# Count of each label type
label_counts = df['type'].value_counts()
print("\nLabel counts:")
print(label_counts)

In [ ]:
# Group the labels (type) to either news or fake news
df['Labels'] = df['type'].apply(lambda x: 'news' if x == 'reliable' else 'fake news')

# Count the distribution of the two groups
unique_labels_grouped = df['Labels'].unique()
print(f"\nUnique label types after grouping: {unique_labels_grouped}")
grouped_labels_counts = df['Labels'].value_counts()
print("\nGrouped label counts:")
print(grouped_labels_counts)

In [ ]:
### Unsure about the rest of the code !!!!

## Part 1 - Task 4

In [ ]:
# Splitting the dataset into training (80%) and temperary splits (20%)
train_df, temp_df = train_test_split(df, test_size=0.20, random_state=42, stratify=df['label'])

# Splitting the temperary split into validation (10%) and test splits (10%)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df['label'])

In [ ]:
# Saving the train, val and test splits into seperate csv's for easier handling later
train_df.to_csv("train.csv", index=False)
val_df.to_csv("val.csv", index=False)
test_df.to_csv("test.csv", index=False)

In [ ]:
# Calculate lenghts of dataset splits
train_len = len(train_df)
val_len = len(val_df)
test_len = len(test_df)

# Calculate the percentage of fake news in the dataset splits
train_perc = (train_df['label'].mean() * 100)
val_perc = (val_df['label'].mean() * 100)
test_perc = (test_df['label'].mean() * 100)

# Printing the statistics
print(f"Training data (80%):    {train_len} rows")
print(f"Validation data (10%): {val_len} rows")
print(f"Test data (10%):        {test_len} rows")
print("-----------------------------------------")
print(f"Fake News percentage in the training data: {train_perc:.1f}%")
print(f"Fake News percentage in the validation data: {val_perc:.1f}%")
print(f"Fake News percentage in the test data: {test_perc:.1f}%")

## Part 2 - Task 1
Grouping done in task 3

## Part 2 - Task 2

In [ ]:
# Creating full texts of the stemmed content
train_texts = train_df['stemmed'].apply(lambda x: " ".join(x))
val_texts = val_df['stemmed'].apply(lambda x: " ".join(x))

# Defining label variables
y_train = train_df['Labels']
y_val = val_df['Labels']

vectorizer = CountVectorizer(vocabulary= vocab_stemmed, max_features=10000)

X_train = vectorizer.fit_transform(train_texts)
X_val = vectorizer.transform(val_texts)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
print(classification_report(y_val, y_pred))

In [ ]:
# Creating vectorizer
vectorizer = CountVectorizer(vocabulary= vocab_stemmed, max_features=10000)

# Transforming the full text
X_train = vectorizer.fit_transform(train_texts)
X_val = vectorizer.transform(val_texts)

In [ ]:
# Creating the logistic regression model
model = LogisticRegression(max_iter=1000, random_state=42)

# Training the model on the trainig data
model.fit(X_train, y_train)

# Predicting on the validation data
y_pred = model.predict(X_val)

# Printing the classification report (focus on F1-score and accuracy)
print(classification_report(y_val, y_pred)) #target_names=['Reliable News', 'Fake News']))

## Part 2 - Task 3

In [ ]:
# Combine titel and stemmed content into one text feature
train_combined_text = train_df['title'].fillna('') + " " + train_df['stemmed'].apply(lambda x: " ".join(x))
val_combined_text = val_df['title'].fillna('') + " " + val_df['stemmed'].apply(lambda x: " ".join(x))

# Vectorize the combined text feature
X_train_text = vectorizer.fit_transform(train_combined_text)
X_val_text = vectorizer.transform(val_combined_text)

In [ ]:
# Make meta-data feature
encoder = OneHotEncoder(handle_unknown='ignore')

# Transform the data
X_train_domain = encoder.fit_transform(train_df[['domain']].fillna('unknown'))
X_val_domain = encoder.transform(val_df[['domain']].fillna('unknown'))

# # Stack features together
X_train_meta = hstack([X_train_text, X_train_domain])
X_val_meta = hstack([X_val_text, X_val_domain])

In [ ]:
# Make logistic regression model for the meta-data
meta_model = LogisticRegression(max_iter=1000, random_state=42)

# Train the meta_model on the training meta-data
meta_model.fit(X_train_meta, y_train)

# Predicting on the validation meta-data
y_pred_meta = meta_model.predict(X_val_meta)

# Printing the classification report (focus on F1-score and accuracy)
print(classification_report(y_val, y_pred_meta)) #target_names=['Reliable News', 'Fake News']))

## Part 3 - Task 1

In [ ]:
# Make tfidf vectorizer
tfidf = TfidfVectorizer(max_features=10000) # Use only the 10,000 most frequent words

# Transform the full text features
train_tfidf = tfidf.fit_transform(train_texts)
val_tfidf = tfidf.transform(val_texts)  

# Make encoder
encoder = LabelEncoder()

# Transform the y_train and y_val to (1, 0)
en_train = encoder.fit_transform(y_train)
en_val = encoder.transform(y_val)

In [ ]:
# Make Neural network prediction model
nn_model = MLPClassifier(
    hidden_layer_sizes=(100,),
    activation='relu',
    solver='adam',
    batch_size='auto', # 256???
    early_stopping=True,
    random_state=42,
    verbose=True
)

# Train the NN model on the weighted and encoded data
nn_model.fit(train_tfidf, en_train)

In [ ]:
# Predicting on the validation data
nn_pred = nn_model.predict(val_tfidf)

# Printing the classification report (focus on F1-score and accuracy)
print(classification_report(en_val, nn_pred))

## Part 4 - Task 1

In [ ]:
# Preparing the test data for the logistic regression model
test_df['Label'] = test_df['type'].apply(lambda x: 'news' if x == 'reliable' else 'fake news')
y_test = test_df['Label']

test_text = test_df['stemmed'].apply(lambda x: " ".join(x))

X_test = vectorizer.transform(test_text)

In [ ]:
# Predicting on the test data with the logistic regression model
test_log = model.predict(X_test)

# Printing the classification report (focus on F1-score and accuracy)
print(classification_report(y_test, test_log))

# Creating confusion matrix
test_log_cm = confusion_matrix(y_test, test_log)

# Printing the results
print("------------------------------")
print(f"TN: {test_log_cm[0][0]}, FP: {test_log_cm[0][1]}, FN: {test_log_cm[1][0]}, TP: {test_log_cm[1][1]}")

In [ ]:
# Preparing the test data for the NN model
X_test = X_test.toarray()  # ????

test_labels = encoder.transform(y_test)

In [ ]:
# Predicting on the test data with the NN model 
test_nn = nn_model.predict(X_test)

# Printing the classification report (focus on F1-score and accuracy)
print(classification_report(test_labels, test_nn))

# Creating confusion matrix
test_nn_cm = confusion_matrix(test_labels, test_nn)

# Printing the results
print("------------------------------")
print(f"TN: {test_nn_cm[0][0]}, FP: {test_nn_cm[0][1]}, FN: {test_nn_cm[1][0]}, TP: {test_nn_cm[1][1]}")


## Part 4 - Task 2

In [ ]:
# Creating column titles to fit the models input format
liar_columns = [
    'id', 'label', 'statement', 'subject', 'speaker', 'speaker_job', 
    'state_info', 'party_affiliation', 'barelytrue_counts', 'false_counts', 
    'halftrue_counts', 'mostlytrue_counts', 'pantsonfire_counts', 'context'
]

# Loading the dataset and adding the column titles
liar = pd.read_csv('test.tsv', sep='\t', names=liar_columns)

# Grouping the labels to either news or fake news
liar['new_label'] = liar['label'].apply(lambda x: 'news' if x == 'true' else 'fake news')

# Printing the first few rows of the dataset
liar.head()

In [ ]:
# Applying cleaning function to the dataset
liar['cleaned'] = liar['statement'].apply(clean_text)

# Stemming the dataset
liar['stemmed'] = liar['cleaned'].apply( lambda x: " ".join([ps.stem(word) for word in x.split()]))

# Creating label variabel
y_liar = liar['new_label']

# Transforming the dataset
X_liar = vectorizer.transform(liar['stemmed'])

In [ ]:
# Predicting on the LIAR dataset with the logistic regression model
liar_log = model.predict(X_liar)

# Printing the classification report (focus on F1-score and accuracy)
print(classification_report(y_liar, liar_log))

# Creating confusion matrix
liar_log_cm = confusion_matrix(y_liar, liar_log)

# Printing the results
print("------------------------------")
print(f"TN: {liar_log_cm[0][0]}, FP: {liar_log_cm[0][1]}, FN: {liar_log_cm[1][0]}, TP: {liar_log_cm[1][1]}")

In [ ]:
# Preparing the data for the NN model
tfidf_liar = tfidf.transform(liar['stemmed'])

liar_labels = encoder.transform(y_liar)

In [ ]:
# Predicting on the LIAR dataset with the NN model
liar_nn = nn_model.predict(tfidf_liar)

# Printing the classification report (focus on F1-score and accuracy)
print(classification_report(liar_labels, liar_nn))

# Creating confusion matrix
liar_nn_cm = confusion_matrix(liar_labels, liar_nn)

# Printing the results
print("------------------------------")
print(f"TN: {liar_nn_cm[0][0]}, FP: {liar_nn_cm[0][1]}, FN: {liar_nn_cm[1][0]}, TP: {liar_nn_cm[1][1]}")

## Part 5

In [ ]:
# Creating function to plot confusion matrices
def plot_cm(cm, title):
    plt.figure()
    plt.imshow(cm)
    
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j], ha='center', va='center')
    
    plt.xticks([0,1], ['Pred Fake', 'Pred Real'])
    plt.yticks([0,1], ['Actual Fake', 'Actual Real'])
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    
    plt.show()

In [ ]:
# Test dataset
plot_cm(test_log_cm, "Test Data - Logistic Regression")
plot_cm(test_nn_cm, "Test Data - Neural Network")

# LIAR dataset
plot_cm(liar_log_cm, "LIAR Data - Logistic Regression")
plot_cm(liar_nn_cm, "LIAR Data - Neural Network")

In [ ]:
# Creating function to get the accuracy of the models
def get_accuracy(cm):
    return (cm[0][0] + cm[1][1]) / np.sum(cm)

# Logistic regression accuracies
log_acc = [
    get_accuracy(test_log_cm),
    get_accuracy(liar_log_cm)
]

# Neural Network accuracies
nn_acc = [
    get_accuracy(test_nn_cm),
    get_accuracy(liar_nn_cm)
]

In [ ]:
# Defining the dataset labels
labels = ['Test Data', 'LIAR Data']

# Defining lenghts of x and width
x = np.arange(len(labels))
width = 0.35

# Plotting the accuracies of the models on the two datasets
plt.figure()
plt.bar(x - width/2, log_acc, width, label='Logistic Regression')
plt.bar(x + width/2, nn_acc, width, label='Neural Network')
plt.xticks(x, labels)
plt.ylabel('Accuracy')
plt.title('Model Comparison Across Datasets')
plt.legend()
plt.show()